[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C30_Agent_Harness_Course/03_llm_adapter/03_llm_adapter.ipynb)

# 03 · LLM 适配器（统一接口：MockLLM ↔ 真实 Claude 一行互换）

目标：设计一个**统一的 LLMClient 接口**，让 `MockLLM` 与真实 `AnthropicLLM` 都实现它；写出真实 Claude 的**消息/工具格式翻译**、**响应解析**、**无 key 自动回退**，并用 `assert` 验证 agent 对二者完全无感。

路线：LLMClient 接口 → MockLLM(统一格式) → 消息/工具格式翻译 → 响应解析 → 无 key 回退 make_llm → agent 对二者无感 → ✏️ 练习 → 📖 答案 → 🧪 真实 Claude 调用胶囊。

> 心智模型：**接口是插座，MockLLM 是电池，真实 Claude 是市电；agent 只认插座**。格式刻意与 Claude Messages API 同构，适配=翻译。纯标准库可跑。

## 1 · LLMClient：所有『大脑』的统一接口

定义接口：`complete(system, messages, tools)` 返回**统一格式**的决策。格式刻意与 Claude 响应同构(`stop_reason`/`tool_calls`/`usage`)。

In [ ]:
class LLMClient:
    '''统一接口。agent 只依赖它，不关心背后是 mock 还是真实 Claude。'''
    def complete(self, system, messages, tools):
        '''返回 {'stop_reason','text','tool_calls','usage'}。'''
        raise NotImplementedError

def is_valid_decision(d):
    '''统一格式的合法性检查(适配器与 mock 都该满足)。'''
    return (isinstance(d, dict)
            and d.get('stop_reason') in ('tool_use', 'end_turn')
            and isinstance(d.get('text', ''), str)
            and isinstance(d.get('tool_calls', []), list)
            and isinstance(d.get('usage', {}), dict))

# 一个手工构造的合法决策
sample = {'stop_reason':'tool_use', 'text':'我先查',
          'tool_calls':[{'id':'1','name':'search','input':{'q':'x'}}],
          'usage':{'input_tokens':10,'output_tokens':5}}
assert is_valid_decision(sample)
assert not is_valid_decision({'stop_reason':'weird'})  # 非法 stop_reason
print('✅ LLMClient 接口与统一决策格式定义完成')

## 2 · MockLLM：确定性、零成本、可断言

MockLLM 不读 history、不思考，按脚本逐条返回统一格式决策。还能伪造 usage(便于后面成本追踪)。

In [ ]:
class MockLLM(LLMClient):
    def __init__(self, script, fake_usage=(10, 5)):
        self.script = list(script); self.calls = 0; self.fake_usage = fake_usage
    def complete(self, system, messages, tools):
        assert self.calls < len(self.script), 'MockLLM 脚本用尽(agent 可能没按预期停)'
        d = dict(self.script[self.calls]); self.calls += 1
        d.setdefault('text', '')
        d.setdefault('tool_calls', [])
        d.setdefault('usage', {'input_tokens':self.fake_usage[0],
                               'output_tokens':self.fake_usage[1]})
        return d

m = MockLLM([
    {'stop_reason':'tool_use','tool_calls':[{'id':'1','name':'search','input':{'q':'x'}}]},
    {'stop_reason':'end_turn','text':'答案'},
])
d0 = m.complete('sys', [], [])
d1 = m.complete('sys', [], [])
assert is_valid_decision(d0) and is_valid_decision(d1)
assert d0['stop_reason'] == 'tool_use' and d1['stop_reason'] == 'end_turn'
assert d0['usage']['input_tokens'] == 10   # 自动补上的 fake usage
print('✅ MockLLM 实现 LLMClient：确定性返回统一格式决策，自动补全 text/tool_calls/usage')

## 3 · 消息格式翻译：本课 history → Messages API messages

真实适配器要把本课 history(含工具结果)翻译成 Claude 的 `messages`。关键：**工具结果要包成 `tool_result` 块放进 user 消息**，助手的工具调用要还原成 `tool_use` 块。我们写翻译函数并验证结构(不需真实 API)。

In [ ]:
def to_api_messages(history):
    '''本课 history -> Claude Messages API 的 messages 列表。
       本课 history 里:
         {'role':'user','content': str 或 [tool_result, ...]}
         {'role':'assistant','content': str, 'tool_calls':[...]}
       翻译成 Claude 的内容块结构。'''
    out = []
    for msg in history:
        role = msg['role']
        content = msg.get('content')
        if role == 'assistant':
            blocks = []
            txt = msg.get('text') or (content if isinstance(content, str) else '')
            if txt:
                blocks.append({'type':'text','text':txt})
            for call in (msg.get('tool_calls') or []):
                blocks.append({'type':'tool_use','id':call['id'],
                               'name':call['name'],'input':call['input']})
            out.append({'role':'assistant','content':blocks or (content or '')})
        else:  # user
            if isinstance(content, list):   # 工具结果列表
                blocks = [{'type':'tool_result','tool_use_id':r['tool_use_id'],
                           'content':r['content'],'is_error':r.get('is_error',False)}
                          for r in content]
                out.append({'role':'user','content':blocks})
            else:
                out.append({'role':'user','content':content})
    return out

hist = [
    {'role':'user','content':'北京天气?'},
    {'role':'assistant','text':'我查一下','tool_calls':[{'id':'t1','name':'get_weather','input':{'city':'北京'}}]},
    {'role':'user','content':[{'tool_use_id':'t1','content':'北京晴','is_error':False}]},
]
api = to_api_messages(hist)
import json
print(json.dumps(api, ensure_ascii=False, indent=1))
# 验证关键结构
assert api[1]['content'][-1]['type'] == 'tool_use'        # 助手的工具调用还原成 tool_use
assert api[1]['content'][-1]['id'] == 't1'
assert api[2]['content'][0]['type'] == 'tool_result'      # 工具结果包成 tool_result
assert api[2]['content'][0]['tool_use_id'] == 't1'        # 靠 id 对回调用
print('✅ 消息翻译正确：tool_use / tool_result 块结构与 Claude 协议一致')

## 4 · 响应解析：Claude content 块 → 统一格式

适配器另一半：把 Claude 响应的 `content` 块(text / tool_use)拆开，连同 `stop_reason`、`usage` 组装成统一格式。
真实 resp 是对象；这里用一个**等价的假对象**模拟其结构，验证解析逻辑(无需真实 API)。

In [ ]:
class _Blk:
    def __init__(self, **kw): self.__dict__.update(kw)
class _Usage:
    def __init__(self, i, o): self.input_tokens = i; self.output_tokens = o
class _Resp:
    def __init__(self, content, stop_reason, usage): 
        self.content = content; self.stop_reason = stop_reason; self.usage = usage

def from_api_response(resp):
    '''Claude messages.create 的响应 -> 本课统一格式决策。'''
    text, tool_calls = '', []
    for blk in resp.content:
        if blk.type == 'text':
            text += blk.text
        elif blk.type == 'tool_use':
            tool_calls.append({'id':blk.id, 'name':blk.name, 'input':blk.input})
    return {'stop_reason':resp.stop_reason, 'text':text, 'tool_calls':tool_calls,
            'usage':{'input_tokens':resp.usage.input_tokens,
                     'output_tokens':resp.usage.output_tokens}}

# 模拟一个『模型想调工具』的真实响应
fake = _Resp(
    content=[_Blk(type='text', text='让我查天气'),
             _Blk(type='tool_use', id='tu_1', name='get_weather', input={'city':'北京'})],
    stop_reason='tool_use',
    usage=_Usage(42, 18))
d = from_api_response(fake)
print(d)
assert is_valid_decision(d)
assert d['stop_reason'] == 'tool_use'
assert d['tool_calls'][0] == {'id':'tu_1','name':'get_weather','input':{'city':'北京'}}
assert d['text'] == '让我查天气'
assert d['usage'] == {'input_tokens':42, 'output_tokens':18}
# 再测一个 end_turn 响应
fake2 = _Resp([_Blk(type='text', text='北京晴 22°C')], 'end_turn', _Usage(50, 10))
d2 = from_api_response(fake2)
assert d2['stop_reason'] == 'end_turn' and d2['tool_calls'] == []
print('✅ 响应解析正确：content 块按类型拆开、stop_reason/usage 映射到统一格式')

## 5 · AnthropicLLM 与 make_llm：无 key 自动回退

把翻译两半合进 `AnthropicLLM`(实现 LLMClient)；再写 `make_llm()`：**有 key 接真模型、没 key 回退 MockLLM，绝不抛异常**。

In [ ]:
class AnthropicLLM(LLMClient):
    '''真实 Claude 适配器。内部用 anthropic SDK 调 Messages API。'''
    def __init__(self, model='claude-opus-4-8', max_tokens=1024):
        import anthropic
        self.client = anthropic.Anthropic()   # 读 ANTHROPIC_API_KEY
        self.model, self.max_tokens = model, max_tokens
    def complete(self, system, messages, tools):
        resp = self.client.messages.create(
            model=self.model, max_tokens=self.max_tokens,
            system=system or 'You are a helpful agent.',
            messages=to_api_messages(messages),
            tools=tools or [])
        return from_api_response(resp)

def make_llm(mock_script, model='claude-opus-4-8'):
    '''有 key 且能 import anthropic -> AnthropicLLM; 否则 -> MockLLM。绝不抛异常。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic  # noqa: F401
            return AnthropicLLM(model)
        except ImportError:
            pass
    return MockLLM(mock_script)

import os
llm = make_llm([{'stop_reason':'end_turn','text':'hi'}])
print('本机使用:', type(llm).__name__)
assert hasattr(llm, 'complete')
assert isinstance(llm, (MockLLM, AnthropicLLM))
if not os.environ.get('ANTHROPIC_API_KEY'):
    assert isinstance(llm, MockLLM), '无 key 必须回退 MockLLM'
print('✅ make_llm 成立：有 key 接真 Claude、没 key 回退 MockLLM，不抛异常')

## 6 · agent 对 llm 完全无感：同一循环，换脑不换码

终极验证：写一个 agent 循环，只依赖 LLMClient 接口。用 MockLLM 驱动它跑通；它**不含一行**与 mock 或 anthropic 耦合的代码——换成 AnthropicLLM 一字不改。

In [ ]:
def dispatch(tools_fns, call):
    fn = tools_fns.get(call['name'])
    if fn is None:
        return {'tool_use_id':call['id'],'content':f'无此工具 {call["name"]}','is_error':True}
    try:
        return {'tool_use_id':call['id'],'content':str(fn(**call['input'])),'is_error':False}
    except Exception as e:
        return {'tool_use_id':call['id'],'content':str(e),'is_error':True}

def run_agent(llm, tools_schema, tools_fns, system, task, max_steps=10):
    '''只依赖 LLMClient.complete —— 对 mock/真实 一视同仁。'''
    history = [{'role':'user','content':task}]
    total_in = total_out = 0
    for step in range(max_steps):
        d = llm.complete(system, history, tools_schema)   # <- 唯一碰『大脑』处
        total_in += d['usage']['input_tokens']; total_out += d['usage']['output_tokens']
        history.append({'role':'assistant','text':d['text'],'tool_calls':d['tool_calls']})
        if d['stop_reason'] == 'end_turn':
            return {'status':'done','answer':d['text'],'tokens':(total_in,total_out)}
        results = [dispatch(tools_fns, c) for c in d['tool_calls']]
        history.append({'role':'user','content':results})
    return {'status':'max_steps','tokens':(total_in,total_out)}

schema = [{'name':'get_weather','description':'查天气',
           'input_schema':{'type':'object','properties':{'city':{'type':'string'}},'required':['city']}}]
fns = {'get_weather': lambda city: f'{city}晴22°C'}
brain = make_llm([
    {'stop_reason':'tool_use','text':'查','tool_calls':[{'id':'1','name':'get_weather','input':{'city':'北京'}}]},
    {'stop_reason':'end_turn','text':'北京今天晴 22°C'},
])
out = run_agent(brain, schema, fns, 'You are a weather agent.', '北京天气?')
print('status:', out['status'], '| answer:', out.get('answer'), '| tokens:', out['tokens'])
assert out['status'] == 'done' or os.environ.get('ANTHROPIC_API_KEY')  # 无 key 必 done
if not os.environ.get('ANTHROPIC_API_KEY'):
    assert '北京' in out['answer']
    assert out['tokens'] == (20, 10)   # 2 次调用 x (10,5)
print('✅ agent 只依赖 LLMClient 接口：MockLLM 跑通，换 AnthropicLLM 一字不改')

---
## ✏️ 练习 1：实现一个会『读历史』的 ScriptedLLM

MockLLM 无视 history。实现 `ScriptedLLM`：它根据**history 里最后一条工具结果**来决定下一步——
若最近一条 user 消息含工具结果且结果含子串 `'晴'`，返回 `end_turn` 答 `'适合出门'`；否则返回调用 `get_weather`。
(这模拟真实 LLM『看结果再决定』，但仍确定性、可断言。)

In [ ]:
class ScriptedLLM(LLMClient):
    def complete(self, system, messages, tools):
        # TODO: 找 messages 里最后一条 role=='user' 且 content 是 list(工具结果)的消息
        #       若其中任一结果的 content 含 '晴' -> 返回 end_turn, text='适合出门'
        #       否则 -> 返回 tool_use 调 get_weather(city='北京')
        #       记得带 usage(随便给, 如 input_tokens=1,output_tokens=1)
        raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
s = ScriptedLLM()
# 初始(无工具结果) -> 应调 get_weather
d0 = s.complete('sys', [{'role':'user','content':'北京天气?'}], [])
assert d0['stop_reason'] == 'tool_use' and d0['tool_calls'][0]['name'] == 'get_weather'
# 给一条含『晴』的工具结果 -> 应收尾
hist = [{'role':'user','content':'北京天气?'},
        {'role':'assistant','text':'查','tool_calls':[{'id':'1','name':'get_weather','input':{'city':'北京'}}]},
        {'role':'user','content':[{'tool_use_id':'1','content':'北京晴 22°C','is_error':False}]}]
d1 = s.complete('sys', hist, [])
assert d1['stop_reason'] == 'end_turn' and d1['text'] == '适合出门'
assert is_valid_decision(d0) and is_valid_decision(d1)
print('✅ 练习 1 通过：ScriptedLLM 能看历史决策，仍确定性可断言')

## ✏️ 练习 2：把工具结果翻译成 tool_result 块

实现 `results_to_user_message(results)`：把本课的工具结果列表 `[{'tool_use_id','content','is_error'}]`，
翻译成一条 Claude user 消息 `{'role':'user','content':[{'type':'tool_result','tool_use_id',...,'is_error'}...]}`。
失败的结果要带 `is_error: True`。

In [ ]:
def results_to_user_message(results):
    # TODO: 返回 {'role':'user','content':[{'type':'tool_result','tool_use_id':r['tool_use_id'],
    #              'content':r['content'],'is_error':r.get('is_error',False)} for r in results]}
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
results = [{'tool_use_id':'A','content':'北京晴','is_error':False},
           {'tool_use_id':'B','content':'工具崩了','is_error':True}]
msg = results_to_user_message(results)
assert msg['role'] == 'user'
assert all(b['type'] == 'tool_result' for b in msg['content'])
assert msg['content'][0]['tool_use_id'] == 'A' and msg['content'][0]['is_error'] is False
assert msg['content'][1]['is_error'] is True
print('✅ 练习 2 通过：工具结果正确翻译成 tool_result 块、保留 is_error')

## ✏️ 练习 3：稳健的响应解析(应对 refusal / 空 content)

真实响应可能 `stop_reason=='refusal'`(安全拒答, content 可能为空)或 content 里有 `thinking` 块。
实现 `from_api_safe(resp)`：在第 4 节基础上——遇到不认识的块 type 就跳过(不崩)；`stop_reason` 原样带回；
若 `stop_reason=='refusal'`，额外在返回 dict 里加 `'refused': True`。

In [ ]:
def from_api_safe(resp):
    # TODO: 遍历 resp.content，只认 text/tool_use，其余(如 thinking)跳过
    #       组装统一格式；若 resp.stop_reason=='refusal' 加 'refused':True
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 含一个 thinking 块(应被跳过)
r1 = _Resp([_Blk(type='thinking', thinking='想...'),
            _Blk(type='text', text='答案')], 'end_turn', _Usage(1,1))
d1 = from_api_safe(r1)
assert d1['text'] == '答案' and d1['stop_reason'] == 'end_turn'
assert d1.get('refused', False) is False
# refusal: content 为空
r2 = _Resp([], 'refusal', _Usage(1,0))
d2 = from_api_safe(r2)
assert d2['refused'] is True and d2['text'] == ''
assert is_valid_decision({**d2, 'stop_reason':'end_turn'})  # 结构仍完整(除 stop_reason)
print('✅ 练习 3 通过：跳过未知块、标记 refusal、不崩')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
class ScriptedLLM(LLMClient):
    def complete(self, system, messages, tools):
        usage = {'input_tokens':1,'output_tokens':1}
        for msg in reversed(messages):
            if msg['role'] == 'user' and isinstance(msg['content'], list):
                if any('晴' in r['content'] for r in msg['content']):
                    return {'stop_reason':'end_turn','text':'适合出门',
                            'tool_calls':[],'usage':usage}
                break
        return {'stop_reason':'tool_use','text':'',
                'tool_calls':[{'id':'1','name':'get_weather','input':{'city':'北京'}}],
                'usage':usage}

In [ ]:
# 练习 2 参考答案
def results_to_user_message(results):
    return {'role':'user',
            'content':[{'type':'tool_result',
                        'tool_use_id':r['tool_use_id'],
                        'content':r['content'],
                        'is_error':r.get('is_error', False)} for r in results]}

In [ ]:
# 练习 3 参考答案
def from_api_safe(resp):
    text, tool_calls = '', []
    for blk in resp.content:
        if blk.type == 'text':
            text += blk.text
        elif blk.type == 'tool_use':
            tool_calls.append({'id':blk.id,'name':blk.name,'input':blk.input})
        # 其余类型(thinking 等)跳过
    out = {'stop_reason':resp.stop_reason, 'text':text, 'tool_calls':tool_calls,
           'usage':{'input_tokens':resp.usage.input_tokens,
                    'output_tokens':resp.usage.output_tokens}}
    if resp.stop_reason == 'refusal':
        out['refused'] = True
    return out

---
## 🧪 真实数据胶囊：真正调用一次 `claude-opus-4-8`

把适配器接到真实 Claude 跑一轮。**有 `ANTHROPIC_API_KEY` + `anthropic` 时真正调用**，看到真实的 `stop_reason`/`usage`/工具调用；**无 key 时自动回退 MockLLM**、用同一份代码跑通。这就是本课『同一份代码、两种读者』的兑现。

> 真实调用推荐开 `thinking={'type':'adaptive'}` 处理复杂任务、用 `max_tokens` 给足余量。

In [ ]:
def call_once(system, user_msg, tools_schema, model='claude-opus-4-8'):
    '''真实调用一次并返回统一格式决策；无 key/无 SDK 回退一个 MockLLM 决策。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            client = anthropic.Anthropic()
            resp = client.messages.create(
                model=model, max_tokens=1024,
                thinking={'type':'adaptive'},   # Claude 4.6+ 自适应思考
                system=system,
                messages=[{'role':'user','content':user_msg}],
                tools=tools_schema)
            return from_api_safe(resp), 'real'
        except Exception as e:
            print('真实调用回退(原因:', type(e).__name__, ')')
    # 回退：MockLLM 给一个合理的 mock 决策
    mock = MockLLM([{'stop_reason':'tool_use',
                     'tool_calls':[{'id':'1','name':tools_schema[0]['name'],
                                    'input':{'city':'北京'}}]}])
    return mock.complete(system, [], tools_schema), 'mock'

schema = [{'name':'get_weather','description':'查询城市天气',
           'input_schema':{'type':'object',
                           'properties':{'city':{'type':'string','description':'城市名'}},
                           'required':['city']}}]
decision, path = call_once('你是天气助手。', '北京今天天气怎样?', schema)
print('路径:', path, '| stop_reason:', decision['stop_reason'])
if decision['tool_calls']:
    print('模型想调:', decision['tool_calls'][0]['name'], decision['tool_calls'][0]['input'])
assert is_valid_decision(decision)
assert path in ('real', 'mock')
print('✅ 真实调用胶囊跑通(有 key 接 Claude、没 key 回退 mock)，返回统一格式决策')

**🧪 胶囊练习**：实现 `estimate_cost(usage, model='claude-opus-4-8')`：按 Claude Opus 4.8 单价(输入 \$5/百万 token、输出 \$25/百万 token)估算一次调用的成本(美元)。返回浮点数。

In [ ]:
def estimate_cost(usage, model='claude-opus-4-8'):
    # TODO: cost = input_tokens/1e6*5 + output_tokens/1e6*25
    raise NotImplementedError

In [ ]:
# 自测
cost = estimate_cost({'input_tokens':1_000_000, 'output_tokens':100_000})
assert abs(cost - (5.0 + 2.5)) < 1e-9   # 1M输入=$5, 0.1M输出=$2.5
small = estimate_cost({'input_tokens':500, 'output_tokens':200})
assert small > 0
print(f'1M 输入 + 0.1M 输出 ≈ ${cost:.2f}')
print('✅ 胶囊练习通过：能据真实单价估算调用成本(模块 04 用它做预算熔断)')

In [ ]:
# 📖 胶囊参考答案
def estimate_cost(usage, model='claude-opus-4-8'):
    PRICES = {'claude-opus-4-8': (5.0, 25.0),    # ($/1M 输入, $/1M 输出)
              'claude-sonnet-4-6': (3.0, 15.0),
              'claude-haiku-4-5': (1.0, 5.0)}
    pin, pout = PRICES.get(model, (5.0, 25.0))
    return usage['input_tokens']/1e6*pin + usage['output_tokens']/1e6*pout

### 小结
- **统一接口 LLMClient**：agent 只认 `complete(system, messages, tools)` 返回的统一格式；不依赖任何具体 SDK。
- **MockLLM** 实现它(确定性、零成本、可断言)；**AnthropicLLM** 实现它(翻译 Claude Messages API)——二者对 agent 完全可互换。
- **适配 = 翻译两半**：`to_api_messages`(history→messages, 工具结果→tool_result 块)、`from_api_response`(content 块→统一格式)。
- **格式同构**：本课的 `tool_calls`/`stop_reason`/`usage` 刻意贴近 Claude 的 `tool_use`/`stop_reason`/`usage`，所以适配只是搬运。
- **make_llm 回退**：有 key 接真 Claude、没 key 回退 MockLLM，**绝不因缺 key 阻断**。
- 官方 `tool_runner` = 手写循环+工具+适配的成品；理解手写版才知道它替你做了什么。

下一站：**模块 04 · 鲁棒性** —— 给适配层和循环加上重试、超时、循环检测、成本追踪，把 demo 变成能跑的系统。